# Inspect alphaZero

## First

### imports and args

In [ ]:
import sys
sys.path.append('..')
sys.path.append('../..')
from Arena import Arena
from KalahGame import KalahGame
from KalahLogic import Board, connect
from utils import *
from MCTS import MCTS  
from KalahPlayers import minmax_tobi, minmax_vince
import time
import multiprocessing
from Coach import Coach
from pytorch.NNetWrapper import NNetWrapper
from pytorch.KalahNNet import KalahNNet
import numpy as np


In [ ]:
args = dotdict({
    'numIters': 1000,
    'numEps': 1,              # Number of complete self-play games to simulate during a new iteration.
    'tempThreshold': 15,        #
    'updateThreshold': 0.6,     # During arena playoff, new neural net will be accepted if threshold or more of games are won.
    'maxlenOfQueue': 200000,    # Number of game examples to train the neural networks.
    'numMCTSSims': 4000,          # Number of games moves for MCTS to simulate.
    'arenaCompare': 40,         # Number of games to play during arena play to determine if new net will be accepted.
    'cpuct': 1,

    'checkpoint': '../best_models',
    'load_model': False,
    'load_folder_file': ('../../best_models/','best.pth.tar'),
    'numItersForTrainExamplesHistory': 20,

})

### inspect

In [ ]:
def inspect_train_samples():
    from Coach import Coach 
    c = Coach(KalahGame(), NNetWrapper(KalahGame()), args)
    c.loadTrainExamples()
    return c.trainExamplesHistory

In [ ]:
history = inspect_train_samples()

In [ ]:
print(len(history)) # different iterations of NN 
print(len(history[0])) # dofferent gamesstates for play with one NN
print(len(history[0][0])) # Board, pi, v
game = 104
print(history[1][game][0]) # Board
print(history[1][game][1]) # pi    
print(history[1][game][2]) # v

### try optimisation of MCTS

In [ ]:
mcts = MCTS(KalahGame(), NNetWrapper(KalahGame()), args)
g = KalahGame()
nnet = NNetWrapper(g)
nnet.load_checkpoint(folder=args.load_folder_file[0], filename='best_64.pth.tar')
player = MCTS(g, nnet, dotdict({ 'numMCTSSims': 4000, 'cpuct': 1.0 }))

board = KalahGame().initial_board
start = time.time()
player.getActionProb(board, 1)
print(time.time() - start)

## Second

### build New NN

#### imports

In [ ]:
import sys
sys.path.append('..')
from utils import *

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim


#### conv Net from original

In [ ]:
class KalahNNet_conv(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_conv, self).__init__()
        self.conv1 = nn.Conv2d(1, args.num_channels, 2, stride=1, padding=1)
        self.conv2 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1, padding=1)
        self.conv3 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1)
        self.conv4 = nn.Conv2d(args.num_channels, args.num_channels, 2, stride=1)

        self.bn1 = nn.BatchNorm2d(args.num_channels)
        self.bn2 = nn.BatchNorm2d(args.num_channels)
        self.bn3 = nn.BatchNorm2d(args.num_channels)
        self.bn4 = nn.BatchNorm2d(args.num_channels)

        self.fc1 = nn.Linear(args.num_channels*(self.board_x)*(self.board_y), 1024)
        self.fc_bn1 = nn.BatchNorm1d(1024)

        self.fc2 = nn.Linear(1024, 512)
        self.fc_bn2 = nn.BatchNorm1d(512)

        self.fc3 = nn.Linear(512, self.action_size)

        self.fc4 = nn.Linear(512, 1)

    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, 1, self.board_x, self.board_y)                # batch_size x 1 x board_x x board_y
        s = F.relu(self.bn1(self.conv1(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn2(self.conv2(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn3(self.conv3(s)))                          # batch_size x num_channels x (board_x-2) x (board_y-2)
        s = F.relu(self.bn4(self.conv4(s)))                          # batch_size x num_channels x (board_x-4) x (board_y-4)
        s = s.view(-1, self.args.num_channels*(self.board_x)*(self.board_y))

        s = F.dropout(F.relu(self.fc_bn1(self.fc1(s))), p=self.args.dropout, training=self.training)  # batch_size x 1024
        s = F.dropout(F.relu(self.fc_bn2(self.fc2(s))), p=self.args.dropout, training=self.training)  # batch_size x 512

        pi = self.fc3(s)                                                                         # batch_size x action_size
        v = self.fc4(s)                                                                          # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)

#### batch normalisation bigger net 

In [ ]:
class KalahNNet_128bn(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_128bn, self).__init__()
        self.fc1 = nn.Linear(self.board_x * self.board_y, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, self.action_size)
        self.fc4 = nn.Linear(128, 1)

        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(128)




    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, self.board_x * self.board_y)                # batch_size x 1 x (board_x * 2)
        s = F.relu(self.fc1(s))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.fc2(s))                          # batch_size x num_channels x board_x x board_y
       
        pi = self.fc3(s)                                                                         # batch_size x action_size
        v = self.fc4(s)                                                                          # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)


### train on previous examples

In [ ]:
import logging
import coloredlogs
from random import shuffle


nnet_args = dotdict({
    'lr': 0.001,
    'dropout': 0.3,
    'epochs': 10,
    'batch_size': 64,
    'cuda': torch.cuda.is_available(),
    'num_channels': 4,
})

log = logging.getLogger(__name__)

def main():
    g = KalahGame()
    nnet = NNetWrapper(g)
    nnet.nnet = KalahNNet_128bn(g, nnet_args)
    history = inspect_train_samples()
    trainExamples = []
    for e in history:
        trainExamples.extend(e)
    shuffle(trainExamples)
    nnet.train(trainExamples)
    nnet.save_checkpoint(folder="../../best_models/", filename='kalah_128bn.pth.tar')


main()

In [ ]:

g = KalahGame()
nnet = NNetWrapper(g)
nnet.nnet = KalahNNet_conv(g, nnet_args)
nnet.load_checkpoint(folder=args.load_folder_file[0], filename='kalah_conv.pth.tar')
player1 = MCTS(g, nnet, args)

pnet = NNetWrapper(g)
pnet.nnet = KalahNNet_128bn(g, nnet_args)
pnet.load_checkpoint(folder=args.load_folder_file[0], filename='kalah_128bn.pth.tar')
player2 = MCTS(g, pnet, args)

arena = Arena(player1=lambda x: np.argmax(player1.getActionProb(x, temp=0)), 
            player2=lambda x: np.argmax(player2.getActionProb(x, temp=0)),
            game=g, 
            display=g.display)

minmax_wins, alpha_zero_wins, draws = arena.playGames(2, verbose=True)
print("minmax_wins", minmax_wins)
print("alpha_zero_wins", alpha_zero_wins)
print("draws", draws)